# Mandatory assignment 3

**GitHub repository:** *Insert the URL to your repository here.*
https://github.com/theokri105/TECH2-H26-assignment3

## About the assignment

- Complete the assignment individually.
- You may use online resources, including generative AI. Include a statement
  explaining how you used AI, or state that you did not use it.

## Submission

- Submit your completed `assignment3.ipynb` on Canvas by
  **October 23, 2026, at 23:59**.

## Submission requirements

- Create a new repository from the template linked on Canvas (click **Use this
  template** > **Create a new repository**) and implement your solution in your
  GitHub repository.
- Use the TECH2 environment from the course.
- Implement your solution directly in this notebook by inserting code and text
  cells. Do not create additional notebooks or files.
- Include all requested numerical results, figures, and short written answers.
- Give each figure a title, label its axes, and include a legend where needed.
- Before you submit, make sure your GitHub repository is **publicly accessible** and include its URL at the
  top of this notebook.
- Before submission, restart the kernel and run all cells in order. Your notebook
  must run without errors and find the CSV without an absolute file path.

## Hints

- Use the material from lectures and workshops 1–3: functions, loops, NumPy,
  pandas, and Matplotlib.
- Compute group statistics using loops; there is no need to use `groupby()`
  (which we cover in lecture 4), but you can do so if you prefer.

***
## Stock market participation in US households

Who participates in the stock market, and how has participation changed over time?
To answer this question, you will analyze data from the
[Survey of Consumer Finances](https://www.federalreserve.gov/econres/scfindex.htm)
(SCF), a representative survey of US households conducted every three years
from 1989 to 2022.

The file `SCF_10pct.csv` contains a 10% sample of the SCF data. The central variables of
interest for this assignment are:

1. `stocks`, which reports the value of directly held stocks (in US dollars); and
2. `equity`, which reports the value of all equity holdings (in US dollars),
   including direct stock ownership and indirect exposure through mutual funds,
   retirement accounts, and other investment vehicles.

We measure stock market participation in two ways:

- **Direct participation:** a household owns individual stocks (`stocks > 0`).
- **Overall participation:** a household holds equity directly or indirectly
  (`equity > 0`).

For each measure, the *participation rate* is the percentage of households
satisfying that condition. We therefore calculate two rates: the **direct
participation rate** and the **overall participation rate**. Direct participants
are included among overall participants, so the two rates should not be added.

## Data description

| Variable | Description |
|----------|-------------|
| `id` | Unique observation identifier |
| `year` | Survey year |
| `age` | Age of the household reference person |
| `educ` | Education level: `1`: No high school; `2`: High school; `3`: Some college; `4`: Bachelor's or higher |
| `stocks` | Value of directly held individual stocks (in US dollars) |
| `equity` | Total directly and indirectly owned stock equity (in US dollars) |

***
## Part 1 — Data preparation

1. Load `SCF_10pct.csv` into a pandas DataFrame. Print the columns contained in
   the DataFrame and their number of non-missing values.
2. Drop all observations that have missing values for any of the variables listed
   in the table above. For each variable, report how many observations you dropped
   and how many remain.
3. Keep observations with `25 <= age <= 89`, including both endpoints.
   Report how many observations you dropped and how many remain.
4. Create two 0/1 indicator columns:
   - `direct`: 1 if the household participates directly (`stocks > 0`),
     0 otherwise.
   - `overall`: 1 if the household participates directly or indirectly
     (`equity > 0`), 0 otherwise.

In [61]:
import pandas as pd

df = pd.read_csv('SCF_10pct.csv')
#df.count()
#df.info()


In [55]:
# Column names
Columns_lst = list(df.columns)
print("Columns:", Columns_lst)

# Number of non-missing values per column
print("\nNon-missing values per column:")
print(df.count())

Columns: ['id', 'year', 'age', 'educ', 'stocks', 'equity']

Non-missing values per column:
id        5808
year      5808
age       5808
educ      5792
stocks    5790
equity    5792
dtype: int64


In [62]:
df = pd.read_csv('SCF_10pct.csv')
na = df.isna().sum()

na_table = pd.DataFrame({
    'NA Values removed': na,
    'Remaining values in colum': len(df) - na
})


na_table.index.name = 'Variable'
print(na_table)
df = df.dropna()

          NA Values removed  Remaining values in colum
Variable                                              
id                        0                       5808
year                      0                       5808
age                       0                       5808
educ                     16                       5792
stocks                   18                       5790
equity                   16                       5792


In [64]:

n_before = len(df)

df = df[(df['age'] >= 25) & (df['age'] <= 89)]

print(f'Dropped {n_before-len(df)} observations and kept: {len(df)} observations')

Dropped 326 observations and kept: 5432 observations


In [75]:
df['direct'] = (df['stocks'] > 0 ).astype(int)
df['overall'] = (df['equity'] > 0).astype(int)

df[['direct', 'overall']]

,direct,overall
0,0,0
1,0,0
2,0,0
4,0,0
5,0,0
...,...,...
5802,0,0
5804,0,0
5805,1,1
5806,1,1


***
## Part 2 — Participation rates

You are now ready to compute the stock market participation rates for the
final sample. You will compute these in two ways: first, by writing a function
that uses a loop to compute the participation rate, and second, by using pandas's
built-in `mean()` method.

1. Write a function `participation_rate(indicator)` that takes a nonempty
   one-dimensional sequence of 0/1 values and returns the fraction of ones
   (in percent).
2. Apply your function to the final sample's `direct` and `overall`
   columns to compute the direct and overall participation rates. Report the
   results.
3. Use pandas's built-in `mean()` method to compute the direct and overall
   participation rates. Report the results.

In [89]:

def participation_rate(indicator):
    count = 0
    total=0
    for i in indicator:
        count += i
    total = count/(len(indicator))
    
    return total

#percent = participation_rate(df['direct'])
#print(f'{percent:%}')

16.955081%


In [ ]:

print(f'{(df['direct'].mean()):%}')
print(f'{(df['overall'].mean()):%}')


16.955081%
49.410898%


In [127]:
direct_p = participation_rate(df['direct'])
overall_p = participation_rate(df['overall'])

print(f'The overall participation rate in the stock market is: {direct_p:.1%}'
      f' and the overall participation is: {overall_p:.1%}.')

The overall participation rate in the stock market is: 17.0% and the overall participation is: 49.4%.


***
## Part 3 — Participation by education

1. Loop over education categories 1–4. For each category, calculate the direct
   participation rate and the overall participation rate (in percent).
2. Display the results, rounding rates to one decimal place.
3. Create a figure with two subplots, one for each rate, containing bar charts
   of participation rates across education categories.
4. In two or three sentences, describe the education pattern.

In [117]:
df.groupby('educ')['1', '2', '3', '4']


print(f'Participation rate in stocks is: {df['direct'].mean()*100:.2f%}')

ValueError: Cannot subset columns with a tuple with more than one element. Use a list instead.

In [135]:
results = []

for i in range(1,5):
    subset = df[df['educ'] == i]
    direct_rate = subset['direct'].mean() * 100
    overall_rate = subset['overall'].mean() * 100
    results.append({'educ': i,
                    'direct': direct_rate,
                    'overall': overall_rate})



educ_rates = pd.DataFrame(results)
educ_rates = educ_rates.set_index('educ').style.format('{:.1f}%')   # use the educ numbers as row labels

educ_rates


,direct,overall
educ,,
1,4.4%,17.3%
2,11.4%,39.9%
3,13.5%,48.3%
4,30.5%,73.5%


In [ ]:
import matplotlib.pyplot as plt

categories = range(1,5)
fig, axes = plt.subplots(1, 2, layout= 'constrained', figsize =(10,4))
xvalues = categories
directvalues = educ_rates['direct']
overallvalues = educ_rates['overall']
axes[0].bar(xvalues, directvalues)
axes[1].bar(xvalues, overallvalues)


In [ ]:
import matplotlib.pyplot as plt

categories = educ_rates.index.astype(str)
fig, axes = plt.subplots(1, 2, layout='constrained', figsize=(10, 4))

axes[0].bar(categories, educ_rates['direct'])
axes[0].set_title('Direct participation rate')
axes[0].set_xlabel('Education category')
axes[0].set_ylabel('Participation rate (%)')

axes[1].bar(categories, educ_rates['overall'])
axes[1].set_title('Overall participation rate')
axes[1].set_xlabel('Education category')
axes[1].set_ylabel('Participation rate (%)')

fig.suptitle('Stock market participation by education')


In [ ]:
# The clear pattern is that higher level of education results in higher degree of investments into stocks and equity.
# As we can se from the data, the overall participation rate i over 4 times higher for bachelor's or higher compared to high school.

***
## Part 4 — Participation over time

1. Loop over years. For each year, compute the direct participation rate and the
   overall participation rate.
2. Create a line plot showing both participation rates over time in a single figure.
3. Report the changes from 1989 to 2022 in the direct and overall participation
   rates, both in percentage points.
4. Describe the main time patterns in two or three sentences.

In [ ]:
years = df['year'].unique()


yearly_p = []
for year in years:
    temporary = df[df['year'] == year]
    direct_yrs = temporary['direct'].mean()
    overall_yrs = temporary['overall'].mean()
    yearly_p.append({'year': year,
                    'Direct':direct_yrs,
                    'overall': overall_yrs})

yearly_investment_rate = pd.DataFrame(yearly_p)
yearly_investment_rate = yearly_investment_rate.set_index('year').style.format('{:.1%}') # I must remember to be careful with style.format as it does wierd things, but works nicely on the end
yearly_investment_rate


first = df[df['year'] == 1989].mean()
first



/var/folders/xz/9mxqsg7s7699znyxmjz7q2lw0000gn/T/ipykernel_70967/3161569793.py:18: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  first = df[df['year'] == 1989].mean([['direct', 'overall']])


TypeError: cannot use 'list' as a dict key (unhashable type: 'list')

***
## AI-use statement

Students must describe their own use of AI here, or state that they did not use it.

For this assignment I have used Ai actively, because I am not on the level of being able to do everything myself. 
It has been used to find issues in the code, suggest methods and provide general useful information,
however I have done all the coding myself.